In [ ]:
import json
import re
import html

INPUT_FILE = "clean_chunks_production.json"
OUTPUT_FILE = "clean_chunks_final_dataset.json"


# ---------------------------------------------------
# SMART TOPIC NORMALIZATION
# ---------------------------------------------------

def normalize_topic(topic, content):

    if not topic:
        return "General"

    topic = html.unescape(topic)

    # remove URLs
    topic = re.sub(r'https?://\S+', '', topic)
    topic = re.sub(r'www\.\S+', '', topic)

    # remove emails
    topic = re.sub(r'\S+@\S+', '', topic)

    # remove phone numbers
    topic = re.sub(r'\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b', '', topic)

    # remove trailing artifacts
    topic = re.sub(r'\b(of|or|ext)\b$', '', topic, flags=re.I)

    # fix ampersands
    topic = topic.replace("&", " and ")

    # remove punctuation artifacts
    topic = re.sub(r'[.,;:]+', ' ', topic)

    # remove double spaces
    topic = re.sub(r'\s+', ' ', topic).strip()

    # special fixes
    topic = topic.replace("Paul Galvin Library", "Paul V. Galvin Library")

    # CONTENTS rename
    if topic.upper() == "CONTENTS":
        topic = "Student Handbook Table of Contents"

    # fallback if topic too long
    if len(topic.split()) > 10:
        topic = " ".join(content.split()[:5])

    # fallback if topic empty
    if len(topic) < 3:
        topic = "General"

    return topic


# ---------------------------------------------------
# CONTENT CLEANING
# ---------------------------------------------------

def clean_content(content):

    if not content:
        return ""

    original = content

    content = html.unescape(content)

    # fix ampersand merge
    content = content.replace("&Vision", "and Vision")

    # remove PDF slash artifacts
    content = re.sub(r'\s+/\s+', ' ', content)

    # normalize whitespace
    content = re.sub(r'\s+', ' ', content).strip()

    if content == "":
        return original

    return content


# ---------------------------------------------------
# MAIN CLEANER
# ---------------------------------------------------

def clean_chunks(chunks):

    cleaned = []

    for chunk in chunks:

        new_chunk = chunk.copy()

        topic = chunk.get("Topic", "")
        content = chunk.get("content", "")

        new_chunk["Topic"] = normalize_topic(topic, content)

        new_chunk["content"] = clean_content(content)

        cleaned.append(new_chunk)

    return cleaned


# ---------------------------------------------------
# RUN
# ---------------------------------------------------

def process():

    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        chunks = json.load(f)

    cleaned = clean_chunks(chunks)

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(cleaned, f, indent=2, ensure_ascii=False)

    print("Final dataset created")
    print("Chunks:", len(cleaned))


process()

Final dataset created
Chunks: 559
